## Mamogram analysis

[Nationwide real-world implementation of AI for cancer detection in population-based mammography screening](https://www.nature.com/articles/s41591-024-03408-6)

Download the praim.csv from here.
https://datadryad.org/dataset/doi:10.5061/dryad.zs7h44jgn

In [1]:
import pandas as pd

mamogram_file = "~/Downloads/praim.csv"



In [2]:
import random
def pipe_filter_exclude_ai_viewer(df):

   return df.query("used_ai_viewer == False").copy()


def compute_pd(df):
   #  first and second read are the same no consense ( 3rd label is NOT used)
   df["pd"] = df[["first_read",	"second_read"]].apply(lambda x: 1.0 if x.iloc[0] == x.iloc[1] else round(2/3,2), axis=1)
   df["pd_majority"] = df["Majority label"]
   
   # Now change the majority label ("NOT Match" when no alignment)  to either sus or normal based on whether the patient was recalled.
   df.loc[df['pd_majority'] == 'NOT MATCH', "pd_majority"] = df.loc[df['pd_majority'] == 'NOT MATCH']["had_recall"].apply(lambda x: "not-normal" if x else "normal" )
   df.loc[df['pd_majority'] == 'suspicious', "pd_majority"] = "not-normal"

   df["all_humans"]= df.apply(lambda x: list([x["first_read"],	x["second_read"],  x['pd_majority']]) if x["Majority label"] == "NOT MATCH"  else  list([x["first_read"],	x["second_read"]]) , axis=1)

   for i in range(5):
        df[f"another_human_{i}"] = df.apply(lambda x: random.choice( x["all_humans"] ), axis=1)
        # Rename value from suspicious to not-normal
        df.loc[df[f"another_human_{i}"] == 'suspicious', f"another_human_{i}"] = "not-normal"



   return df
    
df = pd.read_csv(mamogram_file).pipe(pipe_filter_exclude_ai_viewer).pipe(compute_pd)



In [3]:
df.sample(n=10)

,study_id,cancer_detected,had_recall,used_ai_viewer,reader_set,readers,ai_prediction,had_consensus_conference,had_pre_operation_biopsy,screening_date,...,safety_net_shown,safety_net_accepted,pd,pd_majority,all_humans,another_human_0,another_human_1,another_human_2,another_human_3,another_human_4
220805,220806,False,False,False,rp015,29_41,normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
371222,371223,False,False,False,rp376,48_89_117,normal,False,False,2022-H2,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
99459,99460,False,False,False,rp147,22_111,normal,False,False,2022-H2,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
415936,415937,False,False,False,rp199,58_115,normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
411360,411361,False,False,False,rp147,22_111,normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
144426,144427,False,False,False,rp520,43_85,not-normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
61133,61134,False,False,False,rp110,4_70,normal,False,False,2021-H2,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
290558,290559,False,False,False,rp301,4_47,normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
57109,57110,False,False,False,rp074,91_100,not-normal,False,False,2022-H1,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal
451505,451506,False,False,False,rp509,76_114,normal,False,False,2021-H2,...,False,False,1.0,normal,"[normal, normal]",normal,normal,normal,normal,normal


In [4]:
from sklearn.metrics import classification_report, accuracy_score


def report_by_pd(df, prediction_label = "ai_prediction", predictor_friendly_name="AI"):
    items = []
    scores_dict = classification_report(df["pd_majority"], df[prediction_label],  output_dict=True, zero_division=0.0)["not-normal"]
    scores_dict["PA"]     = "ALL"
    scores_dict["S"]=len(df)
    scores_dict["accuracy"] = accuracy_score(df["pd_majority"], df[prediction_label])

    items.append(scores_dict)
    
    st_groups = list(df["pd"].unique())

    for i in st_groups:



        d_subset  = df.query(f"pd ==  {i}")
        d = classification_report(d_subset["pd_majority"], d_subset[prediction_label],  output_dict=True, zero_division=0.0)      
        if "not-normal"  in d:
            d_treu = d["not-normal"]
            d_treu["PA"]= round(i, 2)
            d_treu["raw_PA"]= i
            d_treu["accuracy"] =accuracy_score(d_subset["pd_majority"], d_subset[prediction_label])
            d_treu["S"]= len(d_subset)
            items.append(d_treu)

    df_scores = pd.DataFrame(items)[["S","PA","accuracy", "precision", "recall", "f1-score", "support"]].round(2)
    df_scores["support"] = df_scores["support"].div(df_scores["S"]).apply(lambda x: "{:.3f}".format(x)) 

    df_scores = df_scores[["PA","S", "support", "f1-score", "precision", "recall", "accuracy"]]
    idx = pd.MultiIndex.from_tuples(
        [
         (predictor_friendly_name, c)  if c in ["accuracy", "precision", "recall", "f1-score"] else  ("common", c)   for c in df_scores.columns
        ],
        names=["predictor", "score_type"]
    )

    df_scores.columns= pd.MultiIndex.from_tuples(idx)
    return df_scores
    
            
df_scores_ai = report_by_pd(df)
df_scores_ai

common                       AI                          
      PA       S support f1-score precision recall accuracy
0    ALL  201079   0.045     0.15      0.08   0.85     0.56
1    1.0  185245   0.029     0.11      0.06   0.89     0.58
2   0.67   15834   0.229     0.39      0.26   0.79     0.42

In [5]:
from functools import reduce

another_human_scores = []
for l in [l for l in list(df.columns) if l.startswith("another_human")]:

     another_human_scores.append(report_by_pd(df, l, l))

# Merge all DataFrames on the 'key_col' column
df_scores_human_all = reduce(lambda left, right: pd.merge(left, right, on=[("common", "PA" ), ("common", "S" ), ("common", "support" )], how='inner'), another_human_scores)

df_scores_human_all

common                 another_human_0                            \
      PA       S support        f1-score precision recall accuracy   
0    ALL  201079   0.045            0.75      0.66   0.87     0.97   
1    1.0  185245   0.029            1.00      1.00   1.00     1.00   
2   0.67   15834   0.229            0.48      0.38   0.67     0.67   

  another_human_1                   ... another_human_2           \
         f1-score precision recall  ...          recall accuracy   
0            0.75      0.66   0.87  ...            0.86     0.97   
1            1.00      1.00   1.00  ...            1.00     1.00   
2            0.48      0.37   0.67  ...            0.65     0.66   

  another_human_3                           another_human_4                   \
         f1-score precision recall accuracy        f1-score precision recall   
0            0.74      0.65   0.86     0.97            0.75      0.66   0.87   
1            1.00      1.00   1.00     1.00            1.00      1.00   1.00   
2            0.47      0.36   0.65     0.66            0.48      0.37   0.68   

            
  accuracy  
0     0.97  
1     1.00  
2     0.67  

[3 rows x 23 columns]

In [6]:
df_scores_human_mean = df_scores_human_all.loc[:, pd.IndexSlice["common", :]].copy()
for c in ["recall", "precision", "f1-score", "accuracy"]:
    df_scores_human_mean.loc[:,("another_human", c)] = df_scores_human_all.loc[:, pd.IndexSlice[:, c]].mean(axis=1)
    df_scores_human_mean.loc[:,("another_human", f"{c}_std")] = df_scores_human_all.loc[:, pd.IndexSlice[:,c]].std(axis=1)
    df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4f}",axis=1)

df_scores_human_mean

/var/folders/7v/5_mr86mx7l9g94fxzdpdx0nw0000gn/T/ipykernel_9866/2414675394.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4f}",axis=1)
/var/folders/7v/5_mr86mx7l9g94fxzdpdx0nw0000gn/T/ipykernel_9866/2414675394.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_scores_human_mean[("another_human", f"fmt_{c}_meanstd")] = df_scores_human_mean[[("another_human", c),("another_human", f"{c}_std")]].apply(lambda x: f"{x[0]:.2f}$\\pm${x[1]:.4

common                 another_human                                \
      PA       S support        recall recall_std fmt_recall_meanstd   
0    ALL  201079   0.045         0.866   0.005477    0.87$\pm$0.0055   
1    1.0  185245   0.029         1.000   0.000000    1.00$\pm$0.0000   
2   0.67   15834   0.229         0.664   0.013416    0.66$\pm$0.0134   

                                                                       \
  precision precision_std fmt_precision_meanstd f1-score f1-score_std   
0     0.658      0.004472       0.66$\pm$0.0045    0.746     0.005477   
1     1.000      0.000000       1.00$\pm$0.0000    1.000     0.000000   
2     0.370      0.007071       0.37$\pm$0.0071    0.476     0.005477   

                                                                   
  fmt_f1-score_meanstd accuracy accuracy_std fmt_accuracy_meanstd  
0      0.75$\pm$0.0055    0.970     0.000000      0.97$\pm$0.0000  
1      1.00$\pm$0.0000    1.000     0.000000      1.00$\pm$0.0000  
2      0.48$\pm$0.0055    0.664     0.005477      0.66$\pm$0.0055

In [7]:
index_columns = [("common", "PA" ), ("common", "S" ), ("common", "support" )]
df_scores_all = df_scores_human_mean.merge(df_scores_ai, on=index_columns)
df_scores_all = df_scores_all.set_index(index_columns)

expected_formula_map = {
    'f1-score': lambda pd, m: (m*pd)/(m*pd + 1-pd),
    "accuracy": lambda pd, m: pd,
    "recall": lambda pd, m: pd,
    "precision": lambda pd, m: (2*m*pd)/(2*m*pd + (1-m)* (1-pd)),
}

for s in ["f1-score", "precision", "recall", "accuracy" ]:
    df_scores_all[("delta_h_ai", s)] = df_scores_all[("another_human", s )]  - df_scores_all[("AI", s )]
    df_scores_all[("Expected", s)] = list(pd.DataFrame(list(df_scores_all.index)).apply(lambda x: expected_formula_map[s](float(x[0]), float(x[2])) if x[0] != 'ALL' else "-", axis=1 ))



df_scores_all

another_human             \
                                                  recall recall_std   
(common, PA) (common, S) (common, support)                            
ALL          201079      0.045                     0.866   0.005477   
1.0          185245      0.029                     1.000   0.000000   
0.67         15834       0.229                     0.664   0.013416   

                                                                         \
                                           fmt_recall_meanstd precision   
(common, PA) (common, S) (common, support)                                
ALL          201079      0.045                0.87$\pm$0.0055     0.658   
1.0          185245      0.029                1.00$\pm$0.0000     1.000   
0.67         15834       0.229                0.66$\pm$0.0134     0.370   

                                                          \
                                           precision_std   
(common, PA) (common, S) (common, support)                 
ALL          201079      0.045                  0.004472   
1.0          185245      0.029                  0.000000   
0.67         15834       0.229                  0.007071   

                                                                           \
                                           fmt_precision_meanstd f1-score   
(common, PA) (common, S) (common, support)                                  
ALL          201079      0.045                   0.66$\pm$0.0045    0.746   
1.0          185245      0.029                   1.00$\pm$0.0000    1.000   
0.67         15834       0.229                   0.37$\pm$0.0071    0.476   

                                                                              \
                                           f1-score_std fmt_f1-score_meanstd   
(common, PA) (common, S) (common, support)                                     
ALL          201079      0.045                 0.005477      0.75$\pm$0.0055   
1.0          185245      0.029                 0.000000      1.00$\pm$0.0000   
0.67         15834       0.229                 0.005477      0.48$\pm$0.0055   

                                                     ...     AI           \
                                           accuracy  ... recall accuracy   
(common, PA) (common, S) (common, support)           ...                   
ALL          201079      0.045                0.970  ...   0.85     0.56   
1.0          185245      0.029                1.000  ...   0.89     0.58   
0.67         15834       0.229                0.664  ...   0.79     0.42   

                                           delta_h_ai  Expected delta_h_ai  \
                                             f1-score  f1-score  precision   
(common, PA) (common, S) (common, support)                                   
ALL          201079      0.045                  0.596         -      0.578   
1.0          185245      0.029                  0.890       1.0      0.940   
0.67         15834       0.229                  0.086  0.317378      0.110   

                                            Expected delta_h_ai Expected  \
                                           precision     recall   recall   
(common, PA) (common, S) (common, support)                                 
ALL          201079      0.045                     -      0.016        -   
1.0          185245      0.029                   1.0      0.110      1.0   
0.67         15834       0.229              0.546705     -0.126     0.67   

                                           delta_h_ai Expected  
                                             accuracy accuracy  
(common, PA) (common, S) (common, support)                      
ALL          201079      0.045                  0.410        -  
1.0          185245      0.029                  0.420      1.0  
0.67         15834       0.229                  0.244     0.67  

[3 rows x 24 columns]

In [8]:
pd.DataFrame(list(df_scores_all.index)).apply(lambda x: x[2] if x[0] != 'ALL' else "-", axis=1)

0        -
1    0.029
2    0.229
dtype: object

In [9]:


def _prep_display(df_scores):
    display_set_cols = []
    cols_score_types = ['f1-score',"precision","recall", "accuracy" ]

    ai_predictor = "AI"
    human_predictor = "another_human"

    for score_type in cols_score_types:
        display_set_cols.append((ai_predictor, score_type))
        display_set_cols.append((human_predictor, f"fmt_{score_type}_meanstd"))
        display_set_cols.append(("delta_h_ai", score_type))
    return df_scores[display_set_cols].sort_index()

_prep_display(df_scores_all )




,,,AI,another_human,delta_h_ai,AI,another_human,delta_h_ai,AI,another_human,delta_h_ai,AI,another_human,delta_h_ai
,,,f1-score,fmt_f1-score_meanstd,f1-score,precision,fmt_precision_meanstd,precision,recall,fmt_recall_meanstd,recall,accuracy,fmt_accuracy_meanstd,accuracy
"(common, PA)","(common, S)","(common, support)",,,,,,,,,,,,
0.67,15834,0.229,0.39,0.48$\pm$0.0055,0.086,0.26,0.37$\pm$0.0071,0.110,0.79,0.66$\pm$0.0134,-0.126,0.42,0.66$\pm$0.0055,0.244
1.0,185245,0.029,0.11,1.00$\pm$0.0000,0.890,0.06,1.00$\pm$0.0000,0.940,0.89,1.00$\pm$0.0000,0.110,0.58,1.00$\pm$0.0000,0.420
ALL,201079,0.045,0.15,0.75$\pm$0.0055,0.596,0.08,0.66$\pm$0.0045,0.578,0.85,0.87$\pm$0.0055,0.016,0.56,0.97$\pm$0.0000,0.410


In [10]:
print(_prep_display(df_scores_all).reset_index().to_latex(float_format="{:.2f}".format, index=False))

\begin{tabular}{lrlrlrrlrrlrrlr}
\toprule
\multicolumn{3}{r}{common} & AI & another_human & delta_h_ai & AI & another_human & delta_h_ai & AI & another_human & delta_h_ai & AI & another_human & delta_h_ai \\
PA & S & support & f1-score & fmt_f1-score_meanstd & f1-score & precision & fmt_precision_meanstd & precision & recall & fmt_recall_meanstd & recall & accuracy & fmt_accuracy_meanstd & accuracy \\
\midrule
0.67 & 15834 & 0.229 & 0.39 & 0.48$\pm$0.0055 & 0.09 & 0.26 & 0.37$\pm$0.0071 & 0.11 & 0.79 & 0.66$\pm$0.0134 & -0.13 & 0.42 & 0.66$\pm$0.0055 & 0.24 \\
1.00 & 185245 & 0.029 & 0.11 & 1.00$\pm$0.0000 & 0.89 & 0.06 & 1.00$\pm$0.0000 & 0.94 & 0.89 & 1.00$\pm$0.0000 & 0.11 & 0.58 & 1.00$\pm$0.0000 & 0.42 \\
ALL & 201079 & 0.045 & 0.15 & 0.75$\pm$0.0055 & 0.60 & 0.08 & 0.66$\pm$0.0045 & 0.58 & 0.85 & 0.87$\pm$0.0055 & 0.02 & 0.56 & 0.97$\pm$0.0000 & 0.41 \\
\bottomrule
\end{tabular}

